# KohakuMoss on Google Colab

Train a LoRA / LyCORIS adapter for MOSS-Image on a Colab GPU runtime.
Runtime > Change runtime type: **RTX PRO 6000 (G4)** or **A100**.
Recipes, measured step times and VRAM per GPU: `docs/guides/colab.md`.

Outputs (checkpoints, samples, exports) go to Google Drive, so they survive a
disconnect; the venv, the weights and the caches live on the VM disk.

In [ ]:
# @title Settings
REPO = "https://github.com/KohakuBlueleaf/KohakuMoss"  # @param {type:"string"}
WEIGHTS_REPO = "OpenMOSS-Team/MOSS-Image-Base"  # @param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/kohakumoss"  # @param {type:"string"}
DATASET = "/content/drive/MyDrive/kohakumoss/datasets/my_dataset"  # @param {type:"string"}
DATASET_SOURCE = "kohya"  # @param ["kohya", "folder", "jsonl", "moss_shard"]
RECIPE = "auto"  # @param ["auto", "recipes_lora_colab_g4.py", "recipes_lora_colab_a100.py", "recipes_lora_24g_bf16.py", "recipes_lycoris_lokr.py", "recipes_lycoris_loha.py", "recipes_dora.py"]
RUN_NAME = "my-lora"  # @param {type:"string"}
STEPS = 2000  # @param {type:"integer"}
SAMPLE_PROMPT = "1girl, solo, looking at viewer, upper body"  # @param {type:"string"}

KM = "/content/KohakuMoss"
VENV = "/content/km-venv"
WEIGHTS = "/content/weights/MOSS-Image"
DATA = "/content/data/" + DATASET.rstrip("/").split("/")[-1]

In [ ]:
# @title Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p {DRIVE_ROOT}/runs {DRIVE_ROOT}/kernel-cache

In [ ]:
# @title Install (venv with the torch build for this GPU's driver, then the doctor report)
!git clone --depth 1 {REPO} {KM} 2>/dev/null || git -C {KM} pull --ff-only
!python {KM}/scripts/colab/setup.py --venv {VENV}

In [ ]:
# @title Download the weights to the VM disk (~26 GB)
!{VENV}/bin/python -c "from huggingface_hub import snapshot_download; snapshot_download('{WEIGHTS_REPO}', local_dir='{WEIGHTS}/base', max_workers=16)"

In [ ]:
# @title Copy the dataset from Drive to the VM disk
!mkdir -p /content/data && rsync -a --info=progress2 {DATASET}/ {DATA}/

In [ ]:
# @title Write the run config
import subprocess

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True,
    text=True,
).stdout
recipe = RECIPE
if recipe == "auto":
    recipe = "recipes_lora_colab_g4.py" if "PRO 6000" in gpu else "recipes_lora_colab_a100.py"
config = f'''from kohakuengine import use_config

use_config("{KM}/configs/moss/recipes/{recipe}")

WEIGHTS = "{WEIGHTS}/base"
DATASETS = [{{"source": "{DATASET_SOURCE}", "path": "{DATA}"}}]
LATENT_CACHE = "online"
TEXT_CACHE = "online"
CACHE_DIR = "/content/cache"
STEPS = {STEPS}
RUN_NAME = "{RUN_NAME}"
OUTPUT_DIR = "{DRIVE_ROOT}/runs"
SAMPLE_PROMPTS = ["{SAMPLE_PROMPT}"]
'''
with open("/content/run_config.py", "w") as f:
    f.write(config)
print(gpu.strip(), "->", recipe)
print(config)

In [ ]:
# @title Train
%cd {KM}
!KOHAKUMOSS_KERNEL_CACHE={DRIVE_ROOT}/kernel-cache {VENV}/bin/kogine run scripts/train/moss.py --config /content/run_config.py

In [ ]:
# @title Sample with the exported adapter
import glob

adapter = sorted(glob.glob(f"{DRIVE_ROOT}/runs/{RUN_NAME}/**/*lycoris*.safetensors", recursive=True))[-1]
!{VENV}/bin/python {KM}/scripts/sample/sample.py --weights {WEIGHTS} --variant base --adapter {adapter} --prompt "{SAMPLE_PROMPT}" --seed 0 --steps 30 --out {DRIVE_ROOT}/runs/{RUN_NAME}/sample_cli

In [ ]:
# @title Release the runtime (stops compute-unit billing)
from google.colab import runtime

runtime.unassign()